# Session 4: The notebook as an instrument

This notebook turns your three Session 3 recordings (CSV files) into **one overlay plot**
and finds when each movement starts. **You do not need to write code.** You only change a few values.

**How to use it**
1. Click into the first grey code cell below and press **Shift + Enter**. The cell runs and the next one is selected.
2. Press **Shift + Enter** again for every cell, **from top to bottom**. While a cell runs you see `[*]` on its left; when it is done you see a number, e.g. `[1]`.
3. The first cell takes up to a minute the first time (the browser loads Python). That is normal.

**The one rule:** only change what is inside a box marked
`>>> YOU MAY CHANGE THIS <<<`. Everything you have to fill in is marked with **TO FILL**.
Change it, then run that cell again (Shift + Enter).

| Cell | What it does | What you may change |
|---|---|---|
| 1 Load | opens your 3 CSV files (if one is missing, it carries on with the others) | the three file names |
| 2 Plot everything | draws all signals (6 or 12) of the first recording | nothing |
| 3 Overlay | draws ONE signal of all recordings in one plot | column name, y-axis label, legend names |
| 4 Movement starts | finds when each movement starts and marks it with lines | rest end time, sensitivity, minimum duration |
| 5 Cut and align | cuts a time window out of each recording and lines them up | start, end, offset per recording, title |
| 6 Save | saves the overlay plot as a PNG picture | the file name |
| 7 Extra (if there is time) | compares the detected starts with your Session 3 table | your Session 3 start times |

**Time:** this session has 60 minutes. Cells 1-6 are the goal. Cell 7 is *extra (if there is time)*.

## Cell 1 — Load the data
Opens your three recordings. If a file is missing you get a message and the notebook **continues with the others**
(for example if you only have recording 1 today).
Replace `NN` by your group number. To practise with the reference file, write `"reference_data.csv"` for recording 1.
Upload files in the file panel on the left (upload button ⬆).

In [ ]:
# ==============================================================
#  CELL 1 — LOAD THE DATA
#  What it does: opens up to three CSV files (tables saved as text) and shows what is inside.
#  Typical problem: 'Recording 2 not found' -> the file name in the box is not exactly
#  the name in the file panel on the left, or the file was not uploaded (yet).
#  If NO file is found at all, the cell stops with a message listing the CSV files that are here.
# ==============================================================

# --------------------------------------------------------------
#  >>> YOU MAY CHANGE THIS <<<
#  FILE_REC1, FILE_REC2, FILE_REC3: the names of your three Session 3 files, in quotes, with ".csv".
#    Change NN to your group number so that the notebook finds your files.
#    TO FILL: replace NN (or write the exact name you see in the file panel on the left).
#    Change a name to "" if you do not have that recording today.
# --------------------------------------------------------------
FILE_REC1 = "reference_rec1.csv"    # recording 1: fixed plan, with taps
FILE_REC2 = "reference_rec2.csv"      # recording 2: your own plan, with taps
FILE_REC3 = "reference_rec3.csv"    # recording 3: your own plan, WITHOUT taps
# --------------------------------------------------------------

# --- From here on: nothing to change ---------------------------

# Load the "toolboxes" (libraries) we need:
#   numpy      -> numbers and arrays
#   pandas     -> reads and handles tables
#   matplotlib -> draws plots
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TIME_COLUMN = "time_s"   # name of the time column in every Capture2Go export (seconds)
FILES = {1: FILE_REC1, 2: FILE_REC2, 3: FILE_REC3}
recs = {}                # recording number -> table with the data

csv_here = [f for f in os.listdir(".") if f.lower().endswith(".csv")]
for n, name in FILES.items():
    if name == "" or not os.path.exists(name):
        print(f"Recording {n} not found, continuing with the others.   (looked for: '{name}')")
        continue
    table = pd.read_csv(name)
    if TIME_COLUMN not in table.columns:
        print(f"Recording {n}: '{name}' has no column '{TIME_COLUMN}' (columns: {list(table.columns)}). Skipping it.")
        continue
    recs[n] = table
    length = table[TIME_COLUMN].iloc[-1] - table[TIME_COLUMN].iloc[0]
    print(f"Recording {n} loaded: {name}  ({len(table)} rows, {len(table.columns)} columns, about {length:.1f} s)")

if not recs:
    raise FileNotFoundError(
        "None of the three files was found.\n"
        f"CSV files that ARE here: {csv_here}\n"
        "-> Check the spelling in the box (capital letters, .csv at the end, NN replaced by your group number) "
        "or upload your files first."
    )

# The signal columns are read from the file itself (6 or 12 columns, any names).
first = min(recs)
signal_columns = [c for c in recs[first].columns if c != TIME_COLUMN]
print()
print("Signal columns you can use:", signal_columns)
print(f"The first 5 rows of recording {first}:")
recs[first].head()

## Cell 2 — Plot everything
Draws every signal of the first recording against time, one small plot per signal, all with the same time axis.
Nothing to change here. Look for the **taps** (short spikes) that you made before each movement in Session 3.
The recordings can have 6 or 12 signals; the plot adapts.

In [ ]:
# ==============================================================
#  CELL 2 — PLOT EVERYTHING
#  What it does: one small plot per signal of the first recording, stacked, same time axis.
#  Nothing to change here. It uses the tables from Cell 1.
#  Typical error: NameError: name 'recs' is not defined -> run Cell 1 first.
# ==============================================================

table = recs[first]
fig_all, axes = plt.subplots(len(signal_columns), 1, sharex=True, squeeze=False,
                             figsize=(10, 1.3 * len(signal_columns)))

for ax, col in zip(axes[:, 0], signal_columns):
    ax.plot(table[TIME_COLUMN], table[col], linewidth=0.8)
    ax.set_ylabel(col, fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1, 0].set_xlabel("Time (s)")
fig_all.suptitle(f"All signals of recording {first}: {FILES[first]}")
plt.tight_layout()
plt.show()

## Cell 3 — Overlay: one signal, all recordings
Draws ONE signal of all your recordings in the same plot, so you can compare them.
Use your labels from the Session 3 worksheet (which sensor, which direction, which unit).
The recordings have different lengths, so for now you see each one as it is. Cells 4 and 5 will cut and align them.

In [ ]:
# ==============================================================
#  CELL 3 — OVERLAY
#  What it does: draws one column of every loaded recording in ONE plot, with a legend.
#  Typical error: KeyError / 'There is no column' -> copy the name EXACTLY from the list printed in Cell 1.
#  Typical problem: the legend shows 'TO FILL: ...' -> you have not written the legend names yet.
# ==============================================================

# --------------------------------------------------------------
#  >>> YOU MAY CHANGE THIS <<<
#  COLUMN_NAME: the EXACT column name, in quotes, e.g. "signal_5" (or "B_y" if you have 12 columns).
#    Change this to another column so that the plot shows another signal. The list is printed in Cell 1.
#  Y_LABEL: what the signal is, with its unit in brackets, e.g. "Gyroscope around y (°/s)".
#    TO FILL: replace the text with your own label (use your S3 labels).
#  LABEL_REC1/2/3: one short name per recording for the legend.
#    TO FILL: describe each recording, e.g. "fixed plan, with taps".
# --------------------------------------------------------------
COLUMN_NAME = "signal_5"
Y_LABEL = "TO FILL: signal name (unit)"
LABEL_REC1 = "TO FILL: describe recording 1"
LABEL_REC2 = "TO FILL: describe recording 2"
LABEL_REC3 = "TO FILL: describe recording 3"
# --------------------------------------------------------------

# --- From here on: nothing to change (except the YOU WRITE THIS box) ---

if COLUMN_NAME not in signal_columns:
    raise KeyError(f"There is no column '{COLUMN_NAME}'. The columns are: {signal_columns}")

labels = {1: LABEL_REC1, 2: LABEL_REC2, 3: LABEL_REC3}
loaded = []   # one entry per recording we can use: (number, table, legend name)
for n in sorted(recs):
    if COLUMN_NAME not in recs[n].columns:
        print(f"Recording {n} has no column '{COLUMN_NAME}', leaving it out.")
        continue
    loaded.append((n, recs[n], labels[n]))

fig_overlay, ax = plt.subplots(figsize=(10, 4))
for n, table, label in loaded:
    ax.plot(table[TIME_COLUMN], table[COLUMN_NAME], linewidth=0.9, label=label)
ax.set_xlabel("Time (s)")
ax.set_ylabel(Y_LABEL)
ax.set_title(COLUMN_NAME + " — all recordings, as recorded")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

if "TO FILL" in Y_LABEL or any("TO FILL" in l for l in labels.values()):
    print("Reminder: replace the 'TO FILL' texts in the box above and run this cell again.")

## Cell 4 — Movement starts
Finds **when each movement starts**: the signal leaves its rest value. The first seconds of the recording
(before `REST_END_TIME`) are the reference for "rest". Short spikes (the **taps**) are ignored because they are
shorter than `MIN_DURATION`. It works also for recording 3 (no taps). You get a list of times and one plot per
recording with a vertical line at every start.
This uses the signal you chose in Cell 3. Compare the times with your Start/End table from Session 3.

In [ ]:
# ==============================================================
#  CELL 4 — MOVEMENT STARTS
#  What it does: finds the start time of every movement and marks it with a line.
#  Typical problems:
#   - Too many starts (noise) -> make SENSITIVITY larger.   - Starts missing -> make it smaller.
#   - Taps counted as movements -> make MIN_DURATION larger.
#   - Nothing found at all -> REST_END_TIME must be a time when your foot was still.
#  Typical error: NameError: name 'loaded' is not defined -> run Cell 3 first.
# ==============================================================

# --------------------------------------------------------------
#  >>> YOU MAY CHANGE THIS <<<
#  REST_END_TIME: second at which the initial rest ends (a number, no quotes).
#    Change this to a time before your first tap or movement so that the rest values are measured correctly.
#  SENSITIVITY: how far the signal must leave its rest value to count as movement (in rest-noise units).
#    Change this to a larger number if too many starts appear, to a smaller one if starts are missing.
#  MIN_DURATION: shortest time (in seconds) that counts as a movement.
#    Change this to a larger number so that short taps are ignored, to a smaller one if movements are missed.
# --------------------------------------------------------------
REST_END_TIME = 2.0
SENSITIVITY = 5
MIN_DURATION = 0.2
# --------------------------------------------------------------

# --- From here on: nothing to change ---------------------------

# The function that finds the starts. You do not need to read it.
def find_onsets(time, signal, rest_end, sensitivity, min_duration, merge_gap=1.5):
    """Return the start times (s) of the movements found in one signal."""
    time = np.asarray(time, dtype=float)
    signal = np.asarray(signal, dtype=float)
    rest = time <= rest_end                      # the samples that count as "rest"
    if rest.sum() < 10:
        raise ValueError(f"Too few samples before REST_END_TIME = {rest_end} s. Use a larger value.")
    rest_mean = signal[rest].mean()              # where the signal sits at rest
    rest_noise = max(signal[rest].std(), 1e-9)   # how much it wiggles at rest
    # True wherever the signal is far away from its rest value
    away = (np.abs(signal - rest_mean) > sensitivity * rest_noise) & ~rest
    # Cut the True-stretches into runs: [first_index, last_index]
    edges = np.flatnonzero(np.diff(np.r_[0, away.astype(int), 0]))
    runs = [[a, b - 1] for a, b in zip(edges[::2], edges[1::2])]
    def join_gaps(runs, max_gap):                # glue runs that are closer than max_gap seconds
        joined = []
        for a, b in runs:
            if joined and time[a] - time[joined[-1][1]] < max_gap:
                joined[-1][1] = b
            else:
                joined.append([a, b])
        return joined
    runs = join_gaps(runs, 0.1)                                          # bridge tiny gaps
    runs = [r for r in runs if time[r[1]] - time[r[0]] >= min_duration]  # drop taps (too short)
    runs = join_gaps(runs, merge_gap)                                    # one movement = one run
    return [float(time[a]) for a, b in runs]

onsets = {}   # recording number -> list of start times (seconds)
fig_onsets, axes = plt.subplots(len(loaded), 1, figsize=(10, 2.8 * len(loaded)), squeeze=False)

for ax, (n, table, label) in zip(axes[:, 0], loaded):
    t = table[TIME_COLUMN].values
    x = table[COLUMN_NAME].values
    ax.plot(t, x, linewidth=0.8)
    ax.set_title(f"Recording {n}: {label}", fontsize=9)
    ax.set_ylabel(COLUMN_NAME, fontsize=8)
    ax.grid(True, alpha=0.3)
    onsets[n] = []
    onsets[n] = find_onsets(t, x, REST_END_TIME, SENSITIVITY, MIN_DURATION)
    for start in onsets[n]:
        ax.axvline(start, color="red", linestyle="--", linewidth=1)
        ax.text(start, ax.get_ylim()[1], f" {start:.2f} s", color="red", fontsize=8, va="top")

axes[-1, 0].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

print("Movement starts (seconds):")
for n, starts in onsets.items():
    print(f"  Recording {n}: {[round(s, 2) for s in starts]}")
if all(len(s) == 0 for s in onsets.values()):
    print("No starts found yet. Check REST_END_TIME and SENSITIVITY (see the typical problems above).")
print("Compare these times with the Start/End table from your Session 3 worksheet.")

## Cell 5 — Cut and align
Cuts a time window out of **each** recording (they last different times and start at different moments) and
draws the cut pieces on top of each other. To compare them, the curves are lined up: either you give an
`OFFSET` per recording, or you set time 0 at the movement start found in Cell 4.
Read the start and end times off the plots of Cells 3 and 4.

In [ ]:
# ==============================================================
#  CELL 5 — CUT AND ALIGN
#  What it does: keeps only the samples between START and END of each recording, shifts them in time
#  and plots them on top of each other.
#  It uses COLUMN_NAME and Y_LABEL from Cell 3 and the starts from Cell 4.
#  Typical problem: a recording is skipped with a message -> its window is outside the recording
#  (look at the time axis in Cell 3 or 4) or START is larger than END.
# ==============================================================

# --------------------------------------------------------------
#  >>> YOU MAY CHANGE THIS <<<
#  RECn_START / RECn_END: seconds where the window of recording n begins / ends (numbers, no quotes).
#    You can edit these to show only the movement you want of each recording. END must be LARGER than START.
#  RECn_OFFSET: seconds that are subtracted from the time axis of recording n.
#    Change this so that the curves start at the same place, e.g. to the start of the movement in recording n.
#  ZERO_AT_ONSET: False or True.
#    Change this to True so that time 0 is the first movement start (from Cell 4) inside each window;
#    the OFFSET values are then ignored.
#  TITLE: what the movement is.
#    TO FILL: write your own title, e.g. "Movement 1: toes up".
# --------------------------------------------------------------
REC1_START = 2.5
REC1_END = 7
REC1_OFFSET = 0

REC2_START = 4
REC2_END = 8.5
REC2_OFFSET = 0

REC3_START = 3
REC3_END = 7
REC3_OFFSET = 0

ZERO_AT_ONSET = False
TITLE = "TO FILL: which movement is this?"
# --------------------------------------------------------------

# --- From here on: nothing to change (except the YOU WRITE THIS box) ---

if "onsets" not in globals():
    print("Cell 4 has not been run yet, so ZERO_AT_ONSET cannot use the movement starts.")
    onsets = {}

settings = {1: (REC1_START, REC1_END, REC1_OFFSET),
            2: (REC2_START, REC2_END, REC2_OFFSET),
            3: (REC3_START, REC3_END, REC3_OFFSET)}

windows = []   # one entry per recording: (number, cut table, legend name, time shift)
for n, table, label in loaded:
    start, end, offset = settings[n]
    if start >= end:
        print(f"Recording {n} skipped: REC{n}_START ({start}) must be smaller than REC{n}_END ({end}).")
        continue
    inside = (table[TIME_COLUMN] >= start) & (table[TIME_COLUMN] <= end)
    piece = table[inside]
    if piece.empty:
        print(f"Recording {n} skipped: no data between {start} s and {end} s. "
              f"This recording runs from {table[TIME_COLUMN].min():.1f} s to {table[TIME_COLUMN].max():.1f} s.")
        continue
    shift = offset
    if ZERO_AT_ONSET:
        starts_inside = [s for s in onsets.get(n, []) if start <= s <= end]
        if starts_inside:
            shift = starts_inside[0]
        else:
            print(f"Recording {n}: no movement start found inside the window, using its OFFSET instead.")
    windows.append((n, piece, label, shift))

if not windows:
    raise ValueError("No recording could be cut. Check the START and END values in the box above.")

x_text = "Time (s), 0 = movement start" if ZERO_AT_ONSET else "Time (s)"

fig_window, ax = plt.subplots(figsize=(10, 4))
for n, piece, label, shift in windows:
    ax.plot(piece[TIME_COLUMN] - shift, piece[COLUMN_NAME], linewidth=1.2, label=label)
ax.set_xlabel(x_text)
ax.set_ylabel(Y_LABEL)
ax.set_title(TITLE)
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

if "TO FILL" in TITLE:
    print("Reminder: replace the 'TO FILL' title in the box above and run this cell again.")

## Cell 6 — Save the picture
Saves the overlay plot from Cell 5 as a PNG picture. It appears in the **file panel on the left**
(click the refresh button ⟳ if you do not see it). Then: **right click on the file → Download**,
and move it from your Downloads folder into your **group folder**.

In [ ]:
# ==============================================================
#  CELL 6 — SAVE THE PICTURE
#  What it does: saves the figure from Cell 5 as a PNG file.
#  Typical error: NameError: name 'fig_window' is not defined -> run Cell 5 first.
#  Remember: the file lives only in this browser. DOWNLOAD it (right click -> Download)
#  and put it in your group folder.
# ==============================================================

# --------------------------------------------------------------
#  >>> YOU MAY CHANGE THIS <<<
#  PNG_NAME: name of the picture, in quotes, ending in ".png".
#    TO FILL: change NN to your group number, no spaces, e.g. "group03_overlay.png".
# --------------------------------------------------------------
PNG_NAME = "groupNN_overlay.png"
# --------------------------------------------------------------

# --- From here on: nothing to change (except the YOU WRITE THIS box) ---

if "fig_window" not in globals():
    raise NameError("There is no plot to save yet. Run Cell 5 first.")

# Make sure the name ends with .png (otherwise add it).
if not PNG_NAME.lower().endswith(".png"):
    PNG_NAME = PNG_NAME + ".png"

# dpi = sharpness of the picture; bbox_inches="tight" cuts white borders.
fig_window.savefig(PNG_NAME, dpi=150, bbox_inches="tight")
print("Saved:", PNG_NAME)
print("Now: file panel on the left -> right click on the file -> Download -> move it to your group folder.")

## Cell 7 — Extra (if there is time): compare with your Session 3 table
Compares the start times found in Cell 4 with the Start times you wrote by hand in Session 3.
This only looks at the signal you chose in Cell 3; a movement may be easier to see in another signal, so not every movement appears.

In [ ]:
# ==============================================================
#  CELL 7 — EXTRA (IF THERE IS TIME): COMPARE WITH SESSION 3
#  What it does: lists detected start times next to the Start times from your S3 worksheet.
#  Typical error: NameError: name 'onsets' is not defined -> run Cell 4 first.
# ==============================================================

# --------------------------------------------------------------
#  >>> YOU MAY CHANGE THIS <<<
#  S3_STARTS: for each recording, the Start times (seconds) from your S3 table, in the order of the movements.
#    TO FILL: write your times in the brackets, e.g.  1: [4.1, 9.0],  so that they are compared with the detected ones.
#    Leave [] for a recording you do not want to compare.
# --------------------------------------------------------------
S3_STARTS = {1: [], 2: [], 3: []}
# --------------------------------------------------------------

# --- From here on: nothing to change (except the YOU WRITE THIS box) ---

if "onsets" not in globals():
    raise NameError("There are no detected starts yet. Run Cell 4 first.")

for n, detected in onsets.items():
    mine = S3_STARTS.get(n, [])
    print(f"Recording {n}:   detected (s)   by hand in S3 (s)   difference (s)")
    if not mine:
        print(f"   (S3_STARTS[{n}] is empty. Detected: {[round(d, 2) for d in detected]})")
        continue
    for det, hand in zip(detected, mine):
        print(f"                {det:8.2f}   {hand:15.2f}   {det - hand:+14.2f}")